# Train the last layer

## Imports 

In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "/home/f24achba/Documents/Edge AI/52-Hertz-KWS/src")))

In [2]:
import tensorflow as tf
import numpy as np
import os, glob
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from preprocessing import AudioPipeline
from preprocessing.dataset_pipeline import load_and_preprocess, make_dataset
from preprocessing.transfer_model import build_transfer_model
from sklearn.metrics import classification_report, confusion_matrix

I0000 00:00:1781084389.394685   49144 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781084389.421223   49144 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1781084390.185093   49144 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## Config

In [3]:
DATA_DIR    = "/home/f24achba/Documents/Edge AI/52-Hertz-KWS/data/raw"
CLASSES     = ["talky", "silence", "unknown"]
BATCH_SIZE  = 16      
EPOCHS   = 25       # backbone gelé
LR       = 1e-3
KERAS_PATH  = "/home/f24achba/Documents/Edge AI/52-Hertz-KWS/src/models/miniresnetv2_s1_64x50_tl.keras"

pipeline = AudioPipeline("/home/f24achba/Documents/Edge AI/52-Hertz-KWS/configuration/config.yaml")

## Chargement des chemins + labels 

In [4]:
all_paths, all_labels = [], []

for label_idx, class_name in enumerate(CLASSES):
    wavs = glob.glob(os.path.join(DATA_DIR, class_name, "*.wav"))
    all_paths  += wavs
    all_labels += [label_idx] * len(wavs)
    print(f"  {class_name:10s} → {len(wavs)} fichiers")

all_paths  = np.array(all_paths)
all_labels = np.array(all_labels)
print(f"\nTotal : {len(all_paths)} samples\n")

  talky      → 295 fichiers
  silence    → 135 fichiers
  unknown    → 326 fichiers

Total : 756 samples



## Split : On utilise 85% pour entraîner car il n'y a plus de test set : la k-fold a déjà joué ce rôle

In [5]:
X_train_final, X_val_final, y_train_final, y_val_final = train_test_split(
    all_paths, all_labels,
    test_size=0.15,
    stratify=all_labels,
    random_state=42
)
print(f"Train : {len(X_train_final)} | Val : {len(X_val_final)}")

Train : 642 | Val : 114


## Class weights 

In [6]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train_final),
    y=y_train_final
)
class_weight_dict_final = dict(enumerate(class_weights))
print(f"\nClass weights : {class_weight_dict_final}")



Class weights : {0: 0.856, 1: 1.8608695652173912, 2: 0.7725631768953068}


## tf.data pipeline 

In [7]:
train_ds = make_dataset(X_train_final, y_train_final, BATCH_SIZE, augment=True)
val_ds   = make_dataset(X_val_final,   y_val_final, BATCH_SIZE,  augment=False)

W0000 00:00:1781084390.899282   49200 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1781084390.937591   49144 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


## model 

In [8]:
tf.keras.backend.clear_session() 
model = build_transfer_model(KERAS_PATH, num_classes=3)
model.compile(
    optimizer=tf.keras.optimizers.Adam(LR),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

## Callbacks 

In [9]:
DEPLOY_PATH = "final_model_deploy.keras"
 
callbacks_final = [
    tf.keras.callbacks.ModelCheckpoint(
        DEPLOY_PATH,
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1
    ),
    ]


## Training

In [10]:
print("\n Entraînement final (modèle de déploiement)")
 
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks_final,
    class_weight=class_weight_dict_final
)


 Entraînement final (modèle de déploiement)
Epoch 1/25


36/41 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3673 - loss: 1.0864
Epoch 1: val_accuracy improved from None to 0.66667, saving model to final_model_deploy.keras
41/41 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.4268 - loss: 1.0925 - val_accuracy: 0.6667 - val_loss: 1.0799 - learning_rate: 0.0010
Epoch 2/25
36/41 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2637 - loss: 1.0840
Epoch 2: val_accuracy improved from 0.66667 to 0.74561, saving model to final_model_deploy.keras
41/41 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.2960 - loss: 1.0779 - val_accuracy: 0.7456 - val_loss: 1.0252 - learning_rate: 0.0010
Epoch 3/25
40/41 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4676 - loss: 1.0896
Epoch 3: val_accuracy improved from 0.74561 to 0.87719, saving model to final_model_deploy.keras
41/41 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.4439 - loss: 1.0552 - val_accuracy: 0.8772 - val_loss: 1.0182 - learning_rate: 0.0010
Epoch 4/25
39/41 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step -

## Evaluation

In [11]:
model_final = tf.keras.models.load_model(DEPLOY_PATH)
loss_f, acc_f = model_final.evaluate(val_ds, verbose=0)
 
print(f"\nVal accuracy finale  : {acc_f:.4f}")
print(f"Attendu (k-fold)     : 0.8889 ± 0.0199")
print(f"Modèle sauvegardé → {DEPLOY_PATH}")


Val accuracy finale  : 0.9474
Attendu (k-fold)     : 0.8889 ± 0.0199
Modèle sauvegardé → final_model_deploy.keras


## Classification report

In [12]:
y_pred_f = np.argmax(model_final.predict(val_ds), axis=1)
y_true_f = np.concatenate([y for _, y in val_ds], axis=0)
print(classification_report(y_true_f, y_pred_f, target_names=CLASSES))

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
              precision    recall  f1-score   support

       talky       0.98      0.98      0.98        45
     silence       0.86      0.90      0.88        20
     unknown       0.96      0.94      0.95        49

    accuracy                           0.95       114
   macro avg       0.93      0.94      0.93       114
weighted avg       0.95      0.95      0.95       114



### Générer le fichier .npz (calibration dataset)

In [13]:
import numpy as np

# Prendre un sous-ensemble représentatif du val set (100-200 samples suffisent)
# Pas besoin de tout le dataset, juste assez pour calibrer les activations
N_CALIB = 200

X_calib_list, y_calib_list = [], []
for x_batch, y_batch in val_ds.unbatch().take(N_CALIB):
    X_calib_list.append(x_batch.numpy())
    y_calib_list.append(y_batch.numpy())

X_calib = np.array(X_calib_list)   # shape : (200, H, W, C) ou (200, T, F)
y_calib = np.array(y_calib_list)   # shape : (200,)

print(f"X_calib shape : {X_calib.shape}  dtype : {X_calib.dtype}")
print(f"y_calib shape : {y_calib.shape}")

# Sauvegarder au format .npz (format attendu par ST Edge AI Developer Cloud)
np.savez("dataset_for_quantization.npz", X=X_calib, y=y_calib)
print("→ dataset_for_quantization.npz sauvegardé")

X_calib shape : (114, 64, 50, 1)  dtype : float32
y_calib shape : (114,)
→ dataset_for_quantization.npz sauvegardé


In [15]:
import tensorflow as tf

# Convertir avec quantification int8
converter = tf.lite.TFLiteConverter.from_keras_model(model_final)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Données de calibration
def representative_dataset():
    for x_batch, _ in val_ds.unbatch().batch(1).take(200):
        yield [x_batch]

converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type  = tf.int8
converter.inference_output_type = tf.int8

tflite_model = converter.convert()

with open("final_model_deploy_int8.tflite", "wb") as f:
    f.write(tflite_model)
print(f"Taille : {len(tflite_model) / 1024:.1f} KB")

INFO:tensorflow:Assets written to: /tmp/tmpbg_8t00r/assets


INFO:tensorflow:Assets written to: /tmp/tmpbg_8t00r/assets


Saved artifact at '/tmp/tmpbg_8t00r'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 64, 50, 1), dtype=tf.float32, name='input_4')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  124969390760608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  124969390752160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  124958853470112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  124958853479088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  124958853476096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  124958853475744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  124958853471344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  124958853333584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  124958853337104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  124958853333056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  124958853339392: Tens

/home/f24achba/Documents/Edge AI/52-Hertz-KWS/52_hz_kws/lib/python3.10/site-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1781092840.930573   49144 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1781092840.930585   49144 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1781092840.930758   49144 reader.cc:83] Reading SavedModel from: /tmp/tmpbg_8t00r
I0000 00:00:1781092840.931487   49144 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1781092840.931494   49144 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpbg_8t00r
I0000 00:00:1781092840.938859   49144 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1781092840.939871   49144 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1781092840.985587   49144 loader.cc:220] Runnin

Taille : 119.0 KB


fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8
W0000 00:00:1781092841.850287   49144 flatbuffer_export.cc:3851] Skipping runtime version metadata in the model. This will be generated by the exporter.
